In [13]:
import re
import json
from pathlib import Path
from typing import List, Dict, Any

In [14]:
# Configuración de rutas
INPUT_FILE = "../../outputs/processed/shpNTpo_txt/all.txt"
OUTPUT_FILE = "genesis_corpus.json"

In [ ]:
# Coleccion de libros
BOOKS_BIBLE = {
    "GEN": "GÉNESIS",
    "RUT": "RUT",
    "JOL": "JOEL",
    "JON": "JONÁS",
    "HEB": "HABACUC",
    "MAL": "MALAQUÍAS",
    "MAT": "MATEO",
    "MRK": "SAN MARCOS",
    "LUK": "SAN LUCAS",
    "1JN": "JUAN",
    "ACT": "HECHOS",
    "ROM": "ROMANOS",
    "1CO": "1 CORINTIOS",
    "2CO": "2 CORINTIOS",
    "GAL": "GÁLATAS",
    "EPH": "EFESIOS",
    "PHP": "FILIPENSES",
    "COL": "COLOSENSES",
    "1TH": "1 TESALONICENSES",
    "2TH": "2 TESALONICENSES",
    "1TI": "1 TIMOTEO",
    "2TI": "2 TIMOTEO",
    "TIT": "TITO",
    "PHM": "FILEMÓN",
    "HEB": "HEBREOS",
    "JAS": "SANTIAGO",
    "1PE": "1 PEDRO",
    "2PE": "2 PEDRO",
    "1JN": "1 JUAN",
    "2JN": "2 JUAN",
    "3JN": "3 JUAN",
    "JUD": "JUDAS",
    "REV": "APOCALIPSIS"
}

In [16]:
def load_text_file(filepath: str) -> str:
    """Carga el archivo de texto con manejo de errores."""
    try:
        with open(filepath, "r", encoding="utf-8-sig") as file:
            return file.read()
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {filepath}")
        raise
    except Exception as e:
        print(f"Error al leer el archivo: {e}")
        raise


def clean_verse_text(text: str) -> str:
    """Limpia el texto del versículo eliminando espacios extras y caracteres no deseados."""
    # Eliminar espacios múltiples
    text = re.sub(r'\s+', ' ', text)
    # Eliminar espacios al inicio y final
    text = text.strip()
    return text


def extract_verses(block: str, book_name: str) -> List[Dict[str, Any]]:
    """Extrae los versículos de un bloque de texto."""
    verses = []
    
    for line in block.splitlines():
        line = line.strip()
        
        # Saltar líneas vacías o que sean solo el nombre del libro
        if not line or line == book_name:
            continue
            
        # Buscar versículos que comienzan con número
        match_verse = re.match(r'^(\d+)\s+(.+)', line)
        if match_verse:
            verse_num = int(match_verse.group(1))
            verse_text = clean_verse_text(match_verse.group(2))
            
            # Solo agregar si el texto no está vacío
            if verse_text:
                verses.append({
                    "verse": verse_num,
                    "text": verse_text
                })
    
    return verses


def process_book(book_key: str, book_name: str, content: str) -> Dict[str, Any]:
    """Procesa un libro completo de la Biblia."""
    # Dividir por capítulos usando el marcador del archivo
    pattern = r"=+\s*" + re.escape(book_key) + r"\d+\.htm\s*=+"
    chapters_raw = re.split(pattern, content)
    
    chapters = []
    
    for block in chapters_raw:
        if not block.strip():
            continue
            
        # Buscar número del capítulo
        match_chapter = re.search(book_name + r"\s+(\d+)", block, re.IGNORECASE)
        if not match_chapter:
            continue
        
        chapter_num = int(match_chapter.group(1))
        verses = extract_verses(block, book_name)
        
        # Solo agregar capítulos con versículos
        if verses:
            chapters.append({
                "chapter": chapter_num,
                "verse_count": len(verses),
                "verses": verses
            })
    
    return {
        "book_key": book_key,
        "book_name": book_name,
        "chapter_count": len(chapters),
        "chapters": chapters
    }


def build_corpus(content: str, books: Dict[str, str]) -> List[Dict[str, Any]]:
    """Construye el corpus completo de la Biblia."""
    corpus = []
    
    for book_key, book_name in books.items():
        print(f"Procesando {book_name}...")
        
        book_data = process_book(book_key, book_name, content)
        
        # Solo agregar libros que tengan capítulos
        if book_data["chapters"]:
            corpus.append(book_data)
            print(f"  ✓ {book_data['chapter_count']} capítulos procesados")
        else:
            print(f"  ⚠ No se encontraron capítulos")
    
    return corpus


def save_corpus(corpus: List[Dict[str, Any]], filepath: str) -> None:
    """Guarda el corpus en formato JSON."""
    try:
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(corpus, f, ensure_ascii=False, indent=2)
        print(f"\n✓ Corpus guardado en: {filepath}")
    except Exception as e:
        print(f"Error al guardar el archivo: {e}")
        raise

In [17]:
def print_corpus_summary(corpus: List[Dict[str, Any]]) -> None:
    """Imprime un resumen del corpus generado."""
    print("\n" + "="*50)
    print("RESUMEN DEL CORPUS")
    print("="*50)
    
    total_chapters = 0
    total_verses = 0
    
    for book in corpus:
        book_verses = sum(ch["verse_count"] for ch in book["chapters"])
        total_chapters += book["chapter_count"]
        total_verses += book_verses
        
        print(f"\n{book['book_name']}:")
        print(f"  - Capítulos: {book['chapter_count']}")
        print(f"  - Versículos: {book_verses}")
    
    print(f"\n{'='*50}")
    print(f"TOTAL: {len(corpus)} libros, {total_chapters} capítulos, {total_verses} versículos")
    print("="*50)

In [18]:
"""Función principal."""
print("Iniciando procesamiento del corpus bíblico...\n")

# Cargar archivo
content = load_text_file(INPUT_FILE)
print(f"✓ Archivo cargado: {len(content)} caracteres\n")

# Construir corpus
corpus = build_corpus(content, BOOKS_BIBLE)

# Mostrar resumen
print_corpus_summary(corpus)

# Guardar resultado
save_corpus(corpus, OUTPUT_FILE)

# Mostrar ejemplo del primer libro
if corpus:
    print("\n" + "="*50)
    print("EJEMPLO DEL PRIMER CAPÍTULO:")
    print("="*50)
    first_book = corpus[0]
    if first_book["chapters"]:
        first_chapter = first_book["chapters"][0]
        print(f"\n{first_book['book_name']} - Capítulo {first_chapter['chapter']}")
        print(f"Versículos: {first_chapter['verse_count']}\n")
        
        # Mostrar los primeros 3 versículos
        for verse in first_chapter["verses"][:3]:
            print(f"{verse['verse']}. {verse['text']}")

Iniciando procesamiento del corpus bíblico...

✓ Archivo cargado: 1694811 caracteres

Procesando 3 JUAN...
  ✓ 1 capítulos procesados

RESUMEN DEL CORPUS

3 JUAN:
  - Capítulos: 1
  - Versículos: 8497

TOTAL: 1 libros, 1 capítulos, 8497 versículos

✓ Corpus guardado en: genesis_corpus.json

EJEMPLO DEL PRIMER CAPÍTULO:

3 JUAN - Capítulo 1
Versículos: 8497

1. Ea riki ja *anciano, enra ja en akonkin noia Gayo saludo bomai.
2. Nokon keen wetsá, enra miki orankin Dios yokatai, ja min shinan meran mia jakonbires ikai keskatiribi, jatíbiain mia jakonbires inon ixon, itan min yoraribi mia jakonbires inon ixon.
3. Jatíribi miibakeax bekana joi akaibaonra ea yoikanke; ja ikon joi chibankin jeneyamai. Mia jaskataibo ninkataxa ea ikonbiresi raroke.
